# Building Regression Models

In the explainer we learned what regression does in principle: it fits a line that minimises the total error, and the resulting equation tells us how much each feature contributes to the prediction. Now we are going to build that model ourselves.

The scenario is straightforward. An estate agency wants an automated valuation tool. We will start with the simplest possible regression — one feature, one line — check that we understand every number in the output, and then add features one at a time to improve accuracy. By the end, we will have a working model and a clear sense of what its coefficients mean in the context of the housing market.

We will:

- Fit a **simple linear regression** (one feature) using scikit-learn
- Interpret the **coefficient** and **intercept** of a regression model
- Plot a **regression line** on a scatter plot
- Fit a **multiple linear regression** (several features)
- Interpret **R²** as a measure of model fit
- Use a fitted model to **make predictions**

In [ ]:
%pip install -q pandas matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression

sns.set_style("whitegrid")

In [ ]:
df = pd.read_csv("../data/property_sales.csv")
print(df.shape)
df.head()

---

## 1. Simple linear regression

In the previous notebook we saw that floor area has the strongest correlation with sale price. That makes it the natural starting point for a model.

**Simple linear regression** models the relationship between one feature (x) and a target (y) as a straight line:

$$y = mx + c$$

where *m* is the **coefficient** (slope) and *c* is the **intercept**.

In scikit-learn, we:

1. Create a `LinearRegression` object.
2. Call `.fit(X, y)` with the feature array `X` (must be 2D) and the target `y`.
3. Read the results from `.coef_` and `.intercept_`.

In [ ]:
# Prepare the data
X_simple = df[["floor_area_sqm"]]  # 2D: double brackets give a DataFrame
y = df["sale_price"]

# Fit the model
model_simple = LinearRegression()
model_simple.fit(X_simple, y)

print(f"Coefficient (slope): {model_simple.coef_[0]:,.1f}")
print(f"Intercept:           {model_simple.intercept_:,.1f}")

### Interpreting the coefficient and intercept

These two numbers are the entire model. Take a moment to read them carefully.

- The **coefficient** tells us how much the predicted price changes for each additional square metre of floor area. If the coefficient is approximately 3,500, the model is saying: each extra square metre adds about £3,500 to the predicted price. That is the kind of plain-language statement the explainer described.
- The **intercept** is the predicted price when floor area is zero. A property with zero floor area does not exist, so the intercept has no real-world meaning on its own. It simply anchors the line in the right position.

### Plotting the regression line

Numbers are useful, but seeing the line on the scatter plot makes it concrete. This is the same "add a trendline" step from the explainer, except now we control the equation.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["floor_area_sqm"], df["sale_price"], alpha=0.2, s=10, label="Actual")

# Generate predictions across the range of floor areas
x_range = pd.DataFrame({"floor_area_sqm": np.linspace(30, 250, 100)})
y_pred_line = model_simple.predict(x_range)

plt.plot(x_range["floor_area_sqm"], y_pred_line, color="red", linewidth=2, label="Regression line")
plt.xlabel("Floor area (sqm)")
plt.ylabel("Sale price (£)")
plt.title("Simple linear regression: price vs floor area")
plt.legend()
plt.tight_layout()
plt.show()

### R² — how much variance does the model explain?

We read about R² in the explainer. Here is where we see it in practice.

**R²** (the coefficient of determination) ranges from 0 to 1. It tells us the proportion of variance in the target that the model explains.

- R² = 0: the model explains none of the variance (predicting the mean would do just as well).
- R² = 1: the model explains all of the variance (perfect predictions).

For a simple model with one feature, R² is the square of the Pearson correlation coefficient we calculated in the previous notebook.

In [ ]:
r2_simple = model_simple.score(X_simple, y)
print(f"R² (simple model): {r2_simple:.3f}")

Floor area alone explains a meaningful share of price variation, but there is plenty left unexplained. The spread around the regression line in the plot above is all the variance that this single-feature model cannot capture. Can we do better by adding more features?

---

## 2. Multiple linear regression

**Multiple linear regression** extends the model to include more than one feature:

$$y = m_1 x_1 + m_2 x_2 + \ldots + m_n x_n + c$$

Each feature gets its own coefficient. The model finds the combination of slopes and intercept that minimises the total squared error across all observations.

This is where the correlation analysis from the previous notebook pays off. We identified several features with meaningful associations to price. Let's use four of them together: `floor_area_sqm`, `bedrooms`, `distance_to_station_km`, and `age_years`.

In [ ]:
features = ["floor_area_sqm", "bedrooms", "distance_to_station_km", "age_years"]
X_multi = df[features]

model_multi = LinearRegression()
model_multi.fit(X_multi, y)

print("Coefficients:")
for name, coef in zip(features, model_multi.coef_):
    print(f"  {name:30s} {coef:>10,.1f}")
print(f"\nIntercept: {model_multi.intercept_:,.1f}")

### Interpreting multiple coefficients

Each coefficient answers a specific question: "if this feature increases by one unit and everything else stays the same, how much does the predicted price change?"

For example:
- If the `bedrooms` coefficient is around 15,000, the model says that adding one bedroom — while keeping floor area, distance, and age the same — is associated with a £15,000 increase in predicted price.
- A negative coefficient on `distance_to_station_km` means that each extra kilometre from the station is associated with a price decrease.

The "holding all else constant" part is the key advantage of multiple regression over simple correlation. In the previous notebook, we saw that bedrooms and floor area are correlated with each other. Multiple regression disentangles them, isolating each feature's individual contribution. That said, "holding all else constant" is a mathematical abstraction. In the real world, we cannot add a bedroom without changing floor area. The model does this separation for us, but keep in mind that the real world is messier than the equation.

In [ ]:
r2_multi = model_multi.score(X_multi, y)
print(f"R² (simple model):   {r2_simple:.3f}")
print(f"R² (multiple model): {r2_multi:.3f}")

Adding more features increases R². But remember the warning from the explainer: R² always goes up when we add variables, even irrelevant ones. Whether this improvement reflects genuine predictive power or just noise is a question for the next notebook, where we will evaluate models on data they have never seen.

---

## 3. Making predictions

A model that cannot make predictions is just a summary. The point of building this regression is to estimate the price of properties the agency has not yet valued. Once the model is fitted, `.predict()` does this in one call.

In [ ]:
# A new property: 85 sqm, 3 bedrooms, 1.2 km from station, 25 years old
new_property = pd.DataFrame({
    "floor_area_sqm": [85],
    "bedrooms": [3],
    "distance_to_station_km": [1.2],
    "age_years": [25],
})

predicted_price = model_multi.predict(new_property)
print(f"Predicted price: £{predicted_price[0]:,.0f}")

We can predict for multiple properties at once, which is what the agency would do in practice.

In [ ]:
new_properties = pd.DataFrame({
    "floor_area_sqm": [50, 85, 120, 180],
    "bedrooms": [1, 3, 4, 5],
    "distance_to_station_km": [0.5, 1.2, 2.0, 3.5],
    "age_years": [5, 25, 60, 10],
})

new_properties["predicted_price"] = model_multi.predict(new_properties[features])
new_properties["predicted_price"] = new_properties["predicted_price"].round(0).astype(int)
new_properties

---

## 4. Inspecting the simple regression with scipy (optional)

scikit-learn gives us the coefficient and R², but not statistical details like the p-value. For simple linear regression, `scipy.stats.linregress` returns everything in one call: slope, intercept, r-value, **p-value**, and standard error. The p-value tells us whether the slope is statistically distinguishable from zero.

In [ ]:
from scipy import stats

result = stats.linregress(df["floor_area_sqm"], df["sale_price"])

print(f"Slope:     {result.slope:,.1f}")
print(f"Intercept: {result.intercept:,.1f}")
print(f"R-value:   {result.rvalue:.4f}")
print(f"P-value:   {result.pvalue:.2e}")
print(f"Std error: {result.stderr:,.2f}")

A very small p-value (close to 0) confirms that the relationship is statistically significant. With 3,000 observations and a genuine linear trend, we would expect the p-value to be extremely small. This is reassuring but not surprising: significance tells us the pattern is real, not that it is strong enough to be useful. R² is the better measure of practical value.

---

## 5. Exercises

### Exercise 1: Simple regression with a different feature

Fit a simple linear regression predicting `sale_price` from `distance_to_station_km`. Print the coefficient and intercept. Calculate R². Then plot the scatter plot with the regression line overlaid. In a comment, state what the coefficient means in practical terms.

In [ ]:
# Your code here


### Exercise 2: Expand the multiple regression

Fit a multiple regression using all of these features: `floor_area_sqm`, `bedrooms`, `bathrooms`, `garden_sqm`, `age_years`, `distance_to_station_km`, `distance_to_school_km`. Print each coefficient with its feature name. Compare the R² to the four-feature model from section 2. Which features have the largest coefficients (in absolute value)?

In [ ]:
# Your code here


### Exercise 3: Predict prices for specific properties

Using your expanded model from Exercise 2, predict the sale price for these three properties:

| floor_area_sqm | bedrooms | bathrooms | garden_sqm | age_years | distance_to_station_km | distance_to_school_km |
|---|---|---|---|---|---|---|
| 45 | 1 | 1 | 0 | 3 | 0.3 | 0.5 |
| 95 | 3 | 2 | 50 | 40 | 1.5 | 1.0 |
| 200 | 5 | 3 | 150 | 120 | 4.0 | 2.5 |

Display the results as a DataFrame with a `predicted_price` column.

In [ ]:
# Your code here


### Exercise 4: Compare simple regressions

Fit a separate simple linear regression for each of these features: `floor_area_sqm`, `bedrooms`, `garden_sqm`, `age_years`, `distance_to_station_km`. For each model, record the R² score. Display the results in a DataFrame sorted from highest to lowest R². Which single feature is the best predictor of price?

In [ ]:
# Your code here


---

## Summary

We have gone from a single correlation number to a working prediction model. Here is what we built:

- A **simple linear regression** with one feature, producing a coefficient we can explain in plain English
- A **multiple linear regression** with several features, where each coefficient describes the effect of one variable while holding the others constant
- **Predictions** for new properties using `.predict()`
- An understanding of **R²** as the proportion of variance explained, with the caveat that it always increases when we add variables

The model is functional, but there is a question we have not answered: does it actually work on properties it has not seen? An R² calculated on the same data the model was trained on is flattering by design. In the next notebook, we will split the data into training and test sets and find out whether these models generalise or whether they have simply memorised the training data.